## Этап обработки данных (Preprocessing Pipeline)
Код разбит на 2 ячейки: 
1. Определение функции предобработки (`preprocessing.py`).
2. Применение функции к данным и сохранение результата.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

class HomeCreditPreprocessor:
    def __init__(self, missing_threshold=0.5, cardinality_threshold=15):
        self.missing_threshold = missing_threshold
        self.cardinality_threshold = cardinality_threshold
        self.cols_to_drop = []
        self.cat_cols_to_ohe = []
        self.num_imputers = {}
        self.cat_imputers = {}
        self.quantiles = {}
        self.scaler = MinMaxScaler()
        self.train_columns = []
        self.income_medians = {}
        self.global_income_median = 0
        self.mean_income = 0
        self.num_cols_to_scale = []
        
    def fit(self, df):
        """Обучается на тренировочных данных (запоминает все параметры)"""
        df = df.copy()
        
        # 1. Ловушка DAYS_EMPLOYED
        if 'DAYS_EMPLOYED' in df.columns:
            df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
            df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace({365243: np.nan})
            
        # 2. Удаление столбцов с >50% пропусков
        self.cols_to_drop = [col for col in df.columns if df[col].isnull().mean() > self.missing_threshold]
        if 'TARGET' in self.cols_to_drop: self.cols_to_drop.remove('TARGET')
        if 'SK_ID_CURR' in self.cols_to_drop: self.cols_to_drop.remove('SK_ID_CURR')
        df = df.drop(columns=self.cols_to_drop)
        
        # 3. Пропуски (запоминаем медианы и моды)
        # ИСПРАВЛЕНИЕ: используем include=[np.number]
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist() 
        cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
        
        for col in ['SK_ID_CURR', 'TARGET']:
            if col in num_cols: num_cols.remove(col)
            
        self.num_imputers = {col: df[col].median() for col in num_cols}
        self.cat_imputers = {col: df[col].mode()[0] if not df[col].mode().empty else 'Unknown' for col in cat_cols}
        
        # Специфичная логика для дохода (по методичке)
        if 'AMT_INCOME_TOTAL' in df.columns and 'NAME_INCOME_TYPE' in df.columns:
            self.income_medians = df.groupby('NAME_INCOME_TYPE')['AMT_INCOME_TOTAL'].median().to_dict()
            self.global_income_median = df['AMT_INCOME_TOTAL'].median()
        else:
            self.income_medians = {}
            self.global_income_median = 0
            
        # 4. Выбросы (запоминаем квантили)
        self.quantiles = {col: (df[col].quantile(0.01), df[col].quantile(0.99)) for col in num_cols}
        
        # 5. Категориальные (разделяем на OHE и удаление)
        self.cat_cols_to_ohe = [col for col in cat_cols if df[col].nunique() <= self.cardinality_threshold]
        self.cols_to_drop.extend([col for col in cat_cols if df[col].nunique() > self.cardinality_threshold])
        
        # 6. Запоминаем средний доход для создания признака INCOME_DEVIATION
        if 'AMT_INCOME_TOTAL' in df.columns:
            self.mean_income = df['AMT_INCOME_TOTAL'].mean()
        
        # Обучаем скейлер и запоминаем финальные колонки
        # Вызываем transform для самого себя, чтобы получить финальный список колонок
        df_temp = self.transform(df)
        self.num_cols_to_scale = [col for col in df_temp.select_dtypes(include=[np.number]).columns if col not in ['SK_ID_CURR', 'TARGET']]
        if len(self.num_cols_to_scale) > 0:
            self.scaler.fit(df_temp[self.num_cols_to_scale])
            
        self.train_columns = df_temp.columns.tolist()
        return self

    def transform(self, df):
        """Применяет правила к train или test"""
        df = df.copy()
        
        # 1. Ловушка DAYS_EMPLOYED
        if 'DAYS_EMPLOYED' in df.columns:
            df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
            df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace({365243: np.nan})
            
        # 2. Удаление
        df = df.drop(columns=[c for c in self.cols_to_drop if c in df.columns], errors='ignore')
        
        # 3. Пропуски
        if 'AMT_INCOME_TOTAL' in df.columns and 'NAME_INCOME_TYPE' in df.columns and self.income_medians:
            df['AMT_INCOME_TOTAL'] = df.apply(
                lambda row: self.income_medians.get(row['NAME_INCOME_TYPE'], self.global_income_median) 
                if pd.isna(row['AMT_INCOME_TOTAL']) else row['AMT_INCOME_TOTAL'], axis=1
            )
            
        for col, val in self.num_imputers.items():
            if col in df.columns: df[col] = df[col].fillna(val)
        for col, val in self.cat_imputers.items():
            if col in df.columns: df[col] = df[col].fillna(val)
            
        # 4. Выбросы (клиппинг)
        for col, (lower, upper) in self.quantiles.items():
            if col in df.columns: df[col] = df[col].clip(lower, upper)
            
        # 5. Новые признаки
        if 'AMT_INCOME_TOTAL' in df.columns:
            df['INCOME_DEVIATION'] = df['AMT_INCOME_TOTAL'] - self.mean_income
        if 'AMT_CREDIT' in df.columns and 'AMT_INCOME_TOTAL' in df.columns:
            df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
            
        # 6. One-Hot Encoding
        df = pd.get_dummies(df, columns=[c for c in self.cat_cols_to_ohe if c in df.columns], drop_first=True, dtype=int)
        
        # 7. Выравнивание колонок (защита от разной размерности)
        if hasattr(self, 'train_columns') and len(self.train_columns) > 0:
            df = df.reindex(columns=self.train_columns, fill_value=0)
            
        # 8. Нормализация
        if hasattr(self, 'num_cols_to_scale') and len(self.num_cols_to_scale) > 0:
            cols_present = [c for c in self.num_cols_to_scale if c in df.columns]
            if cols_present:
                df[cols_present] = self.scaler.transform(df[cols_present])
                
        return df
        
    def fit_transform(self, df, y=None, **fit_params):
        """Удобный метод, объединяющий fit и transform"""
        return self.fit(df).transform(df)

In [2]:
from pathlib import Path
DATA_RAW = Path('data/raw')
DATA_PROCESSED = Path('data/processed')

# Загрузка данных
df = pd.read_csv(DATA_RAW / 'application_train.csv')

print("--- Обучение пайплайна на TRAIN ---")
preprocessor = HomeCreditPreprocessor()
preprocessor.fit(df)
train_processed = preprocessor.transform(df)

print(f"Train размерность: {train_processed.shape}")

print("\n--- Обработка TEST ---")
# Эмулируем тест (1000 строк без TARGET)
test_df = pd.read_csv(DATA_RAW / 'application_test.csv')
# Применяем только transform (он не будет считать nunique заново!)
test_processed = preprocessor.transform(test_df)

print(f"Test размерность: {test_processed.shape}")

# Проверка, что колонки совпадают
assert list(train_processed.columns) == list(test_processed.columns), "Ошибка: колонки не совпадают!"
print("Успех! Размерности и колонки полностью совпадают.")

# Сохранение
train_processed.to_csv(DATA_PROCESSED / 'train_processed.csv', index=False)
test_processed.to_csv(DATA_PROCESSED / 'test_processed.csv', index=False)
print("Файлы сохранены.")

--- Обучение пайплайна на TRAIN ---
Train размерность: (307511, 110)

--- Обработка TEST ---
Test размерность: (48744, 110)
Успех! Размерности и колонки полностью совпадают.
Файлы сохранены.
